# §2.2 Background Micro-Benchmark Figures

Generates figures supporting `2_Background.tex` §2.2:

- **(a)** Bandwidth ladder — DRAM vs NVMe asymmetry
- **(b)** Memory wall — execution time across 27/28/29 qubits, OOM transition

(GPU utilization timeline was attempted from `tegrastats` logs but the 100 ms sampling could not capture sub-burst GPU activity at 29q — both schemes report ~0% mean utilization. Cell (c) is kept for parsing/inspection but not used as a figure.)

All output PDFs go to `../Figures/`.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
MICRO = ROOT / 'micro_benchmark_results'
FIGS  = ROOT / 'Figures'
FIGS.mkdir(exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'axes.linewidth': 0.8,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'legend.frameon': False,
    'legend.fontsize': 9,
})

C_DRAM_GPU  = '#5B8DEF'
C_DRAM_UMA  = '#92B5F2'
C_DRAM_CPU  = '#BFD4F2'
C_NVME_SEQ  = '#F2A65A'
C_NVME_RAND = '#E07A36'
C_OOM       = '#C8324A'
C_BMQ       = '#E07A36'
C_EDGE      = '#3CA374'
C_NATIVE    = '#5B8DEF'
C_UVM       = '#9974D1'

## (a) Bandwidth Ladder

DRAM tier (GPU D2D / UMA H2D&D2H / CPU memcpy) vs NVMe tier (sequential 256K, random 4K).

In [ ]:
bw = [
    ('GPU DRAM\n(D2D)',       22.9 * 1024,  C_DRAM_GPU),
    ('UMA H2D',               25.8 * 1024,  C_DRAM_UMA),
    ('UMA D2H',               25.0 * 1024,  C_DRAM_UMA),
    ('CPU DRAM\n(memcpy)',     6.0 * 1024,  C_DRAM_CPU),
    ('NVMe Seq Read\n(256K)',  1387,        C_NVME_SEQ),
    ('NVMe Seq Write\n(256K)', 1009,        C_NVME_SEQ),
    ('NVMe Rand Read\n(4K)',   62,          C_NVME_RAND),
    ('NVMe Rand Write\n(4K)',  129,         C_NVME_RAND),
]
labels = [b[0] for b in bw]
values = [b[1] for b in bw]
colors = [b[2] for b in bw]

fig, ax = plt.subplots(figsize=(8.2, 3.6))
x = np.arange(len(bw))
bars = ax.bar(x, values, color=colors, edgecolor='black',
              linewidth=0.6, width=0.7)
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8.5, rotation=20, ha='right')
ax.set_ylabel('Bandwidth (MB/s, log)')
ax.set_ylim(30, 200000)
ax.grid(axis='y', linestyle=':', alpha=0.5, which='both')
ax.set_axisbelow(True)

for b, v in zip(bars, values):
    lab = f'{v/1024:.1f} GB/s' if v >= 1024 else f'{v:.0f} MB/s'
    ax.text(b.get_x() + b.get_width()/2, v * 1.18, lab,
            ha='center', va='bottom', fontsize=8.3, fontweight='bold')

ax.axvline(3.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)
ax.text(1.5, 130000, 'DRAM tier', ha='center', fontsize=10.5,
        fontweight='bold', color='#1f4f8b')
ax.text(5.5, 130000, 'NVMe tier', ha='center', fontsize=10.5,
        fontweight='bold', color='#a04a14')

ax.annotate('', xy=(6.6, 95), xytext=(6.6, 22000),
            arrowprops=dict(arrowstyle='->', color='#7a0f0f', lw=1.4))
ax.text(7.5, 1500, '~400$\\times$\nslower\n(rand 4K)',
        ha='center', va='center', fontsize=8.5, color='#7a0f0f',
        fontweight='bold')
ax.text(4.5, 50000, 'DRAM $\\rightarrow$ NVMe Seq:  ~18$\\times$ slower',
        ha='center', va='center', fontsize=9, color='#7a0f0f',
        fontweight='bold')

plt.tight_layout()
plt.savefig(FIGS / 'fig_bg_bandwidth_ladder.pdf', bbox_inches='tight')
plt.savefig(FIGS / 'fig_bg_bandwidth_ladder.png', dpi=180, bbox_inches='tight')
plt.show()

## (b) Memory Wall (27 → 28 → 29 qubits)

Loads `results.json`. cuQuantum Native and UVM are refused at 29q (state > 80% VRAM). BMQSim-like and EdgeQuantum auto-fallback to tiered mode and survive.

In [ ]:
with (MICRO / 'results.json').open() as f:
    results = json.load(f)

schemes = ['cuQuantum_Native', 'cuQuantum_UVM', 'BMQSim-like', 'EdgeQuantum']
scheme_color = {
    'cuQuantum_Native': C_NATIVE, 'cuQuantum_UVM': C_UVM,
    'BMQSim-like': C_BMQ, 'EdgeQuantum': C_EDGE,
}
scheme_label = {
    'cuQuantum_Native': 'cuQuantum (Native)',
    'cuQuantum_UVM':    'cuQuantum (UVM)',
    'BMQSim-like':      'BMQSim-like',
    'EdgeQuantum':      'EdgeQuantum',
}
qubits = sorted(set(r['qubits'] for r in results['results']))
data = {s: {q: None for q in qubits} for s in schemes}
for r in results['results']:
    if r['success']:
        data[r['name']][r['qubits']] = r['sim_time']

fig, ax = plt.subplots(figsize=(6.8, 3.6))
x = np.array(qubits)
width = 0.18
offsets = {s: (i - 1.5) * width for i, s in enumerate(schemes)}

for s in schemes:
    ys = [data[s][q] for q in qubits]
    xs = x + offsets[s]
    for xi, yi in zip(xs, ys):
        if yi is None:
            ax.bar(xi, 1e-4, width=width, color='white', edgecolor=C_OOM,
                   hatch='///', linewidth=1.2)
            ax.text(xi, 3e-4, 'OOM', ha='center', va='bottom',
                    fontsize=8, color=C_OOM, fontweight='bold', rotation=90)
        else:
            ax.bar(xi, yi, width=width, color=scheme_color[s],
                   edgecolor='black', linewidth=0.5)

handles = [plt.Rectangle((0, 0), 1, 1, facecolor=scheme_color[s],
                          edgecolor='black', linewidth=0.5)
           for s in schemes]
handles.append(plt.Rectangle((0, 0), 1, 1, facecolor='white',
                             edgecolor=C_OOM, hatch='///', linewidth=1.0))
leglabels = [scheme_label[s] for s in schemes] + ['OOM (refused)']
ax.legend(handles, leglabels, loc='upper left', ncol=2, fontsize=8.5)

ax.set_yscale('log')
ax.set_ylim(1e-4, 300)
ax.set_xticks(x)
ax.set_xticklabels([f'{q}q\n({(2**q*8)/(1024**3):.0f} GB)' for q in qubits])
ax.set_ylabel('Simulation time (s, log)')
ax.set_xlabel('Number of qubits (state-vector size)')
ax.grid(axis='y', linestyle=':', alpha=0.5, which='both')
ax.set_axisbelow(True)

ax.axvspan(28.5, 29.5, color=C_OOM, alpha=0.07)
ax.text(29.0, 130, 'Memory Wall\n(state > 80% VRAM)',
        ha='center', va='top', fontsize=9, color=C_OOM,
        fontweight='bold')

y28 = data['EdgeQuantum'][28]
y29 = data['EdgeQuantum'][29]
ax.annotate('', xy=(29 + offsets['EdgeQuantum'], y29 * 0.6),
            xytext=(28 + offsets['EdgeQuantum'], y28 * 1.5),
            arrowprops=dict(arrowstyle='->', color='#444444',
                            lw=1.0, ls='--'))
ax.text(28.5 + offsets['EdgeQuantum'], 0.4, '~50,000$\\times$',
        ha='center', va='center', fontsize=8.5,
        color='#444444', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGS / 'fig_bg_memory_wall.pdf', bbox_inches='tight')
plt.savefig(FIGS / 'fig_bg_memory_wall.png', dpi=180, bbox_inches='tight')
plt.show()

## (c) GPU Utilization Timeline (inconclusive)

Parsed for completeness. `tegrastats` reports GR3D_FREQ at 100 ms granularity, which is too coarse to catch the GPU bursts at 29q (both schemes report 0% mean). Skipped as a figure.

In [ ]:
GR3D_RE = re.compile(r'GR3D_FREQ\s+(\d+)%')

def parse_tegrastats(path):
    gpu = []
    with path.open() as f:
        for line in f:
            m = GR3D_RE.search(line)
            if m:
                gpu.append(int(m.group(1)))
    rel = np.arange(len(gpu)) * 0.1  # tegrastats sampling = 100ms
    return rel, np.array(gpu)

for label, fname in [('BMQSim-like 29q', 'tegrastats_BMQSim-like_29q.log'),
                     ('EdgeQuantum 29q', 'tegrastats_EdgeQuantum_29q.log')]:
    t, g = parse_tegrastats(MICRO / fname)
    print(f'{label}: {len(t)} samples, last t = {t[-1]:.1f}s, '
          f'mean GPU = {g.mean():.1f}%, max = {g.max()}%')

## Output

- `Figures/fig_bg_bandwidth_ladder.{pdf,png}` — §2.2 Figure 1
- `Figures/fig_bg_memory_wall.{pdf,png}` — §2.2 Figure 2

Reference both from `2_Background.tex` §2.2.